In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator
from scipy.stats import norm, chi2, false_discovery_control
from adjustText import adjust_text

In [ ]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Sex Heterogeneity: Male vs Female METAL Meta-Analysis

For every variant present in both the male and female METAL meta-analyses, the per-sex betas are
compared with a two-sample Z-test

`r` is the correlation between the male and female estimates. The two strata share no samples, so `r = 0`
by default (the null-variant Z correlation is printed below as a check).

Inputs are the files written by `run_metal()` in `05_run_METAL.ipynb`:
`Effect` / `StdErr` / `Pvalue` (fixed) and `EffectARE` / `StdErrARE` / `PvalueARE` (random), oriented to `Allele1`.

**X coding:** the sex-stratified GWAS used `--xchr-model 2`, so male non-PAR dosages are 0..2 like the females. Male and female
betas are therefore on the same per-allele scale (the full dosage-compensation model), and no rescaling is applied.

### Parameters

In [ ]:
label = 'GP2_FIGS_EUR_AJ'     # 'GP2_EUR_AJ' or 'GP2_FIGS_EUR_AJ'
regions = ['non_PAR']         # ['non_PAR', 'PAR']
model = 'fixed'               # 'fixed' (Effect/StdErr) or 'random' (EffectARE/StdErrARE)

i_max = 80                    # drop variants with HetISq >= i_max within either sex
min_het_df = 1 if label == 'GP2_EUR_AJ' else 2   # >= 2 (GP2) / >= 3 (GP2 + FIGS) contributing studies
maf = None                    # e.g. 0.01 to drop rare variants (on Freq1 in each sex)

r = 0.0                       # male/female estimate correlation (0 = no sample overlap)
alpha = 0.05                  # family-wise alpha for Bonferroni / FDR

# Targeted test: lead variants significant in either sex, Bonferroni over that set only
lead_p = 1e-5                 # 5e-8 or 1e-5
win_kb = {'non_PAR': 500, 'PAR': 100}   # PAR1 is only ~2.7 Mb

metal_dir = f'{wd}/working/R{rel}_prep/XWAS/meta/METAL/{label}'
out_dir = f'{metal_dir}/sex_het'
os.makedirs(out_dir, exist_ok=True)

### Load METAL results

In [ ]:
EFFECT_COLS = {
    'fixed':  ('Effect', 'StdErr', 'Pvalue'),
    'random': ('EffectARE', 'StdErrARE', 'PvalueARE'),
}
KEEP_COLS = ['CHR', 'POS', 'MarkerName', 'REF', 'ALT', 'Allele1', 'Allele2', 'Freq1',
             'Direction', 'HetISq', 'HetPVal', 'TotalSampleSize']


def merged_path(sex, region, label):
    return f'{metal_dir}/locuszoom/merged/{label}_{region}_{sex}_filtered.tsv.gz'


def load_metal(sex, region, label, model='fixed', i_max=80, min_het_df=1, maf=None):
    # Annotated METAL output for one sex/region, standardized to BETA / SE / P
    df = pd.read_csv(merged_path(sex, region, label), sep='\t', compression='gzip')
    b, se, p = EFFECT_COLS[model]
    n0 = len(df)

    # same filters as 05_run_METAL.ipynb; unaligned rows keep METAL's alphabetical allele order
    keep = (df.HetISq < i_max) & (df.HetDf >= min_het_df)
    if 'ALIGNED' in df.columns:
        keep &= df.ALIGNED.astype(bool)
    if maf is not None:
        keep &= df.Freq1.between(maf, 1 - maf)
    keep &= df[[b, se, p]].notna().all(axis=1) & (df[se] > 0)
    df = df[keep]

    out = df[KEEP_COLS].copy()
    out['BETA'], out['SE'], out['P'] = df[b], df[se], df[p]

    # duplicated markers (e.g. multi-allelic splits): keep the most significant row
    n_dup = int(out.MarkerName.duplicated().sum())
    if n_dup:
        out = out.sort_values('P').drop_duplicates('MarkerName').sort_values('POS')

    print(f'{sex:>6} {region:<7}: {n0:,} -> {len(out):,} variants after filters'
          + (f' ({n_dup:,} duplicate markers dropped)' if n_dup else ''))
    return out.reset_index(drop=True)


def add_rsids(df, sex, region, label):
    # rsIDs live in the LocusZoom-merged files from 05_run_METAL.ipynb, if they've been made
    path = merged_path(sex, region, label)
    if not os.path.exists(path):
        df['rsID'] = np.nan
        return df
    ids = pd.read_csv(path, sep='\t', compression='gzip', usecols=['MarkerName', 'rsid'])
    ids = ids.drop_duplicates('MarkerName').rename(columns={'rsid': 'rsID'})
    return df.merge(ids, on='MarkerName', how='left')

### Align sexes and run the two-sample Z-test

In [ ]:
def merge_sexes(male, female):
    # Inner join on MarkerName (chr:pos:sorted alleles) and put female betas on the male Allele1
    mf = male.merge(female, on='MarkerName', suffixes=('_M', '_F'))
    same = mf.Allele1_M.eq(mf.Allele1_F) & mf.Allele2_M.eq(mf.Allele2_F)
    swap = mf.Allele1_M.eq(mf.Allele2_F) & mf.Allele2_M.eq(mf.Allele1_F)

    mf.loc[swap, 'BETA_F'] = -mf.loc[swap, 'BETA_F']
    mf.loc[swap, 'Freq1_F'] = 1 - mf.loc[swap, 'Freq1_F']
    mf.loc[swap, 'Direction_F'] = mf.loc[swap, 'Direction_F'].str.translate(str.maketrans('+-', '-+'))

    print(f'male: {len(male):,}  female: {len(female):,}  overlap: {len(mf):,}')
    print(f'  same Allele1: {same.sum():,}   flipped: {swap.sum():,}   allele mismatch (dropped): {(~(same | swap)).sum():,}')
    mf = mf[same | swap]

    # one copy of the variant description columns
    mf = mf.rename(columns={f'{c}_M': c for c in ['CHR', 'POS', 'REF', 'ALT', 'Allele1', 'Allele2']})
    mf = mf.drop(columns=[f'{c}_F' for c in ['CHR', 'POS', 'REF', 'ALT', 'Allele1', 'Allele2']])
    return mf.sort_values('POS').reset_index(drop=True)


def null_z_corr(mf, p_min=0.1):
    # Correlation of male/female Z among null variants: ~0 when the strata share no samples
    null = mf[(mf.P_M > p_min) & (mf.P_F > p_min)]
    return np.corrcoef(null.BETA_M / null.SE_M, null.BETA_F / null.SE_F)[0, 1]


def sex_diff_test(mf, r=0.0):
    # Two-sample Z-test on the betas, with Bonferroni and BH-FDR across every variant tested
    mf['BETA_DIFF'] = mf.BETA_M - mf.BETA_F
    mf['SE_DIFF'] = np.sqrt(mf.SE_M**2 + mf.SE_F**2 - 2 * r * mf.SE_M * mf.SE_F)
    mf['Z_DIFF'] = mf.BETA_DIFF / mf.SE_DIFF
    mf['P_DIFF'] = 2 * norm.sf(np.abs(mf.Z_DIFF))
    mf['P_DIFF_BONF'] = np.minimum(mf.P_DIFF * len(mf), 1)
    mf['Q_DIFF'] = false_discovery_control(mf.P_DIFF, method='bh')
    mf['OR_M'], mf['OR_F'] = np.exp(mf.BETA_M), np.exp(mf.BETA_F)
    # which sex carries the larger absolute effect
    mf['STRONGER_IN'] = np.where(mf.BETA_M.abs() > mf.BETA_F.abs(), 'male', 'female')
    return mf


def lambda_gc(p):
    return chi2.isf(np.median(p), 1) / chi2.ppf(0.5, 1)


def get_leads(d, p_col, sig, win):
    # Most significant variant per locus, loci separated by > win bp
    s = d[d[p_col] < sig].sort_values(p_col)
    keep = []
    for _, row in s.iterrows():
        if all(abs(row.POS - k.POS) > win for k in keep):
            keep.append(row)
    return pd.DataFrame(keep, columns=d.columns)

In [ ]:
results, targeted = {}, {}

for region in regions:
    print(f'\n===== {label} {region} ({model} effects) =====')
    male = load_metal('male', region, label, model, i_max, min_het_df, maf)
    female = load_metal('female', region, label, model, i_max, min_het_df, maf)

    mf = merge_sexes(male, female)
    mf = add_rsids(mf, 'male', region, label)
    print(f'null Z correlation (male vs female): {null_z_corr(mf):.4f}   (r used: {r})')

    mf = sex_diff_test(mf, r=r)
    print(f'variants tested: {len(mf):,}   Bonferroni threshold: {alpha / len(mf):.2e}   '
          f'lambda_GC(P_diff): {lambda_gc(mf.P_DIFF):.4f}')
    print(f'  Bonferroni sig: {(mf.P_DIFF_BONF < alpha).sum():,}   FDR < {alpha}: {(mf.Q_DIFF < alpha).sum():,}')

    # Targeted: lead variants (P < lead_p in either sex), Bonferroni over the number of leads
    mf['P_MIN'] = mf[['P_M', 'P_F']].min(axis=1)
    leads = get_leads(mf, 'P_MIN', lead_p, win_kb[region] * 1000).copy()
    if len(leads):
        leads['P_DIFF_BONF_LEADS'] = np.minimum(leads.P_DIFF * len(leads), 1)
    print(f'  leads (P < {lead_p:g} in either sex): {len(leads):,}   '
          f'sex-different after Bonferroni over leads: {(leads.get("P_DIFF_BONF_LEADS", pd.Series(dtype=float)) < alpha).sum():,}')

    results[region], targeted[region] = mf, leads

### Significant sex differences

In [ ]:
SHOW = ['rsID', 'CHR', 'POS', 'MarkerName', 'Allele1', 'Allele2', 'Freq1_M', 'Freq1_F',
        'OR_M', 'P_M', 'OR_F', 'P_F', 'BETA_DIFF', 'SE_DIFF', 'Z_DIFF', 'P_DIFF', 'Q_DIFF', 'STRONGER_IN']

for region, mf in results.items():
    print(f'--- {region}: all tested variants, Bonferroni P_diff < {alpha} ---')
    display(mf[mf.P_DIFF_BONF < alpha].sort_values('P_DIFF')[SHOW])

    print(f'--- {region}: all tested variants, FDR q < {alpha} ---')
    display(mf[mf.Q_DIFF < alpha].sort_values('P_DIFF')[SHOW])

In [ ]:
for region, leads in targeted.items():
    print(f'--- {region}: lead variants (P < {lead_p:g} in either sex) ---')
    if len(leads):
        display(leads.sort_values('P_DIFF')[SHOW + ['P_DIFF_BONF_LEADS']])

In [ ]:
# Save full results and the significant subsets
for region in regions:
    mf, leads = results[region], targeted[region]
    prefix = f'{out_dir}/{label}_{region}_{model}_sex_het'
    mf.to_csv(f'{prefix}_all.tsv.gz', sep='\t', index=False, compression='gzip')
    mf[(mf.P_DIFF_BONF < alpha) | (mf.Q_DIFF < alpha)].sort_values('P_DIFF').to_csv(
        f'{prefix}_sig.tsv', sep='\t', index=False)
    leads.to_csv(f'{prefix}_leads.tsv', sep='\t', index=False)
    print(f'saved {prefix}_*')

### Beta-beta plots (lead variants)

In [ ]:
def beta_beta_plot(leads, title, alpha=0.05, annotate=10, out=None):
    # Male vs female beta at the lead variants; red = sex-different after Bonferroni over the leads
    sig = leads.P_DIFF_BONF_LEADS < alpha
    fig, ax = plt.subplots(figsize=(7, 7))

    for mask, color, lab in [(~sig, '#597FBD', 'Not different'), (sig, '#CB132D', f'P_diff (Bonf.) < {alpha}')]:
        d = leads[mask]
        ax.errorbar(d.BETA_M, d.BETA_F, xerr=1.96 * d.SE_M, yerr=1.96 * d.SE_F, fmt='o', ms=5,
                    color=color, ecolor=color, elinewidth=0.8, alpha=0.8, label=f'{lab} (n={mask.sum()})')

    lim = np.nanmax(np.abs(np.r_[leads.BETA_M + 1.96 * leads.SE_M, leads.BETA_F + 1.96 * leads.SE_F,
                                 leads.BETA_M - 1.96 * leads.SE_M, leads.BETA_F - 1.96 * leads.SE_F])) * 1.1
    ax.plot([-lim, lim], [-lim, lim], ls='--', c='grey', lw=1)
    ax.axhline(0, c='lightgrey', lw=0.8)
    ax.axvline(0, c='lightgrey', lw=0.8)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

    rho = np.corrcoef(leads.BETA_M, leads.BETA_F)[0, 1] if len(leads) > 2 else np.nan
    concord = (np.sign(leads.BETA_M) == np.sign(leads.BETA_F)).mean()
    ax.text(0.03, 0.97, f'r = {rho:.3f}\nsign concordance = {concord:.1%}\nn = {len(leads)}',
            transform=ax.transAxes, va='top', fontsize=10)

    texts = []
    for _, l in leads.nsmallest(annotate, 'P_DIFF').iterrows():
        lab = l.rsID if pd.notna(l.rsID) else f'X:{int(l.POS)}'
        texts.append(ax.text(l.BETA_M, l.BETA_F, lab, fontsize=8))
    if texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))

    ax.set_xlabel(r'Male $\beta$ (95% CI)')
    ax.set_ylabel(r'Female $\beta$ (95% CI)')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=9, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    if out:
        fig.savefig(out, dpi=600, bbox_inches='tight')
    plt.show()
    return fig


for region, leads in targeted.items():
    if len(leads) == 0:
        print(f'{region}: no lead variants at P < {lead_p:g}')
        continue
    beta_beta_plot(leads, f'{label} {region}: male vs female ({model})', alpha=alpha,
                   out=f'{out_dir}/{label}_{region}_{model}_beta_beta.png')

### Manhattan + QQ of the sex-difference P-values

In [ ]:
BLUE, RED = '#597FBD', '#CB132D'   # GWASLab default colors


def plot_diff_mqq(mf, title, alpha=0.05, out=None):
    bonf = alpha / len(mf)
    y = -np.log10(mf.P_DIFF.values)
    x = mf.POS.values / 1e6
    hit = mf.P_DIFF.values < bonf
    ymax = max(-np.log10(bonf), y.max()) * 1.15

    fig, (ax_m, ax_q) = plt.subplots(1, 2, figsize=(16, 5.2),
                                     gridspec_kw={'wspace': 0.14, 'width_ratios': [3.2, 1]})
    size = np.clip(1 + y * 1.2, 1, 12)
    ax_m.scatter(x[~hit], y[~hit], s=size[~hit], c=BLUE, lw=0, rasterized=True)
    ax_m.scatter(x[hit], y[hit], s=size[hit] * 1.5, c=RED, lw=0, rasterized=True, zorder=3)
    ax_m.axhline(-np.log10(bonf), ls=(0, (5, 3)), lw=2, c='grey', zorder=2)
    ax_m.set_ylim(0, ymax)
    ax_m.set_xlabel('Chromosome X position (Mb)', fontsize=12)
    ax_m.set_ylabel(r'$-\log_{10}(P_{diff})$', fontsize=12)
    ax_m.set_title(title)

    p = np.sort(mf.P_DIFF.values)
    obs = -np.log10(p)
    exp = -np.log10((np.arange(1, len(p) + 1) - 0.5) / len(p))
    ax_q.scatter(exp, obs, s=np.clip(1 + obs * 1.5, 1, 14), c=BLUE, lw=0, rasterized=True)
    ax_q.plot([0, exp.max()], [0, exp.max()], ls='--', c='grey', lw=1.5)
    ax_q.text(0.1, 0.97, rf'$\lambda_{{\mathrm{{GC}}}}$ = {lambda_gc(p):.4f}',
              transform=ax_q.transAxes, va='top', fontsize=11)
    ax_q.set_ylim(0, ymax)
    ax_q.set_xlim(0, None)
    ax_q.set_xlabel(r'Expected $-\log_{10}(P)$', fontsize=12)
    ax_q.set_ylabel(r'Observed $-\log_{10}(P)$', fontsize=12)

    for ax in (ax_m, ax_q):
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        ax.spines[['top', 'right']].set_visible(False)
    if out:
        fig.savefig(out, dpi=600, bbox_inches='tight')
    plt.show()
    return fig


for region, mf in results.items():
    plot_diff_mqq(mf, f'{label} {region}: male vs female beta difference ({model})', alpha=alpha,
                  out=f'{out_dir}/{label}_{region}_{model}_sex_diff_mqq.png')

### Look up specific variants

In [ ]:
region = 'non_PAR'
rsIDs = ['']

mf = results[region]
display(mf[mf.rsID.isin(rsIDs)][SHOW])